In [4]:
# 目前VGG效果最好
import torch
from torch import nn
from d2l import torch as d2l
from torchinfo import summary

In [2]:
def nin_block(in_channel, out_channel, kernel_size, stride, padding):  # 全部一样的卷积层
    return nn.Sequential(
        nn.Conv2d(in_channel, out_channel, kernel_size, stride=stride, padding=padding),
        nn.ReLU(),
        nn.Conv2d(in_channel, out_channel, kernel_size, stride=stride, padding=padding),
        nn.Conv2d(in_channel, out_channel, kernel_size, stride=stride, padding=padding)
    )

In [9]:
net = nn.Sequential(
    nin_block(1,96,kernel_size=11,stride=4,padding=0),  # 没有填充
    nn.MaxPool2d(3,stride=2),
    nin_block(96,256,kernel_size=5,stride=1,padding=2),  # 很神奇的参数 我总感觉有联系
    nn.MaxPool2d(3,stride=2),
    nin_block(256,384,kernel_size=3,stride=1,padding=1),
    nn.MaxPool2d(3,stride=2),
    nn.Dropout(0.5),
    # 标签数量为10
    nin_block(384,10,kernel_size=3,stride=1,padding=1),
    nn.AdaptiveAvgPool2d((1,1)),  # 四维转换为二维输出  自适应平均池化层 强制输出
    nn.Flatten(),  # 展平
)

In [11]:
# summary(net,input_size=(1,1,224,224))
X = torch.randn(size=(1, 1, 224, 224))
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__, X.shape)

RuntimeError: Given groups=1, weight of size [96, 1, 11, 11], expected input[1, 96, 54, 54] to have 1 channels, but got 96 channels instead